# Задание 3. Подсчёт автомобилей и оценка загруженности парковки по видео с камеры наблюдения с использованием YOLOv8 и трекинга ByteTrack

| | |
|---|---|
| **Входные данные** | видеозапись парковки с камеры наблюдения или дрона (открытые видео, например Pexels, или датасет VisDrone) |
| **Что определяет система** | число автомобилей в кадре и в зоне парковки, число уникальных машин за запись, загруженность в % от вместимости |
| **Модель** | YOLOv8n, предобученная на COCO (классы car, motorcycle, bus, truck), без дообучения; трекинг объектов ByteTrack |
| **Оценка** | сравнение с ручным подсчётом на 20–30 контрольных кадрах: MAE, MAPE, доля кадров с ошибкой ≤ 1 машины |

Ноутбук работает и в Google Colab, и локально. В Colab включите GPU: **Среда выполнения → Сменить среду выполнения → T4 GPU**.

## 1. Подготовка

In [ ]:
import sys, os, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("bigdata-practice").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/farzXr/bigdata-practice.git"], check=True)
    os.chdir("bigdata-practice/task3_parking_yolo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

from count_cars import pick_device
print("Рабочая папка:", Path.cwd())
print("Устройство:", pick_device())

## 2. Видео

Положите видео парковки в папку `videos/` (локально) или загрузите его ячейкой ниже (в Colab).
Подойдёт любая запись со статичной камеры, где сверху или под углом видны машины: например, на [Pexels](https://www.pexels.com/search/videos/parking%20lot/) по запросу «parking lot» или «aerial parking».
Для отчёта хватит 30–60 секунд.

In [ ]:
VIDEO_DIR = Path("videos")
VIDEO_DIR.mkdir(exist_ok=True)

if IN_COLAB and not any(VIDEO_DIR.glob("*.mp4")):
    from google.colab import files
    for name, data in files.upload().items():
        (VIDEO_DIR / name).write_bytes(data)

videos = sorted(VIDEO_DIR.glob("*.mp4"))
assert videos, "Положите .mp4 в папку videos/"
VIDEO = videos[0]
print("Видео:", VIDEO)

In [ ]:
# Первый кадр: по нему удобно выбрать зону парковки (ROI) в долях кадра
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(str(VIDEO))
ok, frame = cap.read()
cap.release()
h, w = frame.shape[:2]
plt.figure(figsize=(11, 6))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), extent=[0, 1, 1, 0])
plt.xticks([i / 10 for i in range(11)]); plt.yticks([i / 10 for i in range(11)])
plt.grid(color="yellow", alpha=0.5)
plt.title(f"Первый кадр {w}x{h}: координаты в долях для параметра ROI")
plt.show()

## 3. Подсчёт машин

Параметры:
- `ROI` — зона парковки в долях кадра; `None` — считать во всём кадре;
- `CAPACITY` — число мест, чтобы посчитать загруженность в %; `None` — не считать;
- `STRIDE` — обрабатывать каждый N-й кадр: 1 — все, 2–3 — быстрее на процессоре.

In [ ]:
ROI = None            # например: "0.05,0.35;0.95,0.35;0.95,0.98;0.05,0.98"
CAPACITY = None       # например: 40
STRIDE = 1 if pick_device() != "cpu" else 2

cmd = [sys.executable, "count_cars.py", "--source", str(VIDEO), "--stride", str(STRIDE)]
if ROI:
    cmd += ["--roi", ROI]
if CAPACITY:
    cmd += ["--capacity", str(CAPACITY)]
subprocess.run(cmd, check=True)
OUT = Path("out") / VIDEO.stem

## 4. Результаты

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

summary = json.loads((OUT / "summary.json").read_text(encoding="utf-8"))
display(pd.Series(summary, name="значение").to_frame())
display(Image(str(OUT / "occupancy.png")))

In [ ]:
# Несколько кадров из размеченного видео
cap = cv2.VideoCapture(str(OUT / "annotated.mp4"))
n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, idx in zip(axes, [0, n // 2, max(n - 1, 0)]):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, img = cap.read()
    if ok:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"кадр {idx * STRIDE}")
    ax.axis("off")
cap.release()
plt.tight_layout()
plt.show()

counts = pd.read_csv(OUT / "counts.csv")
counts.describe().round(2)

## 5. Оценка точности

1. Ячейка ниже выгружает 25 контрольных кадров в `out/<видео>/frames/` и создаёт `ground_truth.csv`.
2. Посчитайте машины на каждом кадре вручную (в зоне ROI, если она задана) и впишите числа в колонку `true_count`.
3. Запустите следующую ячейку, она посчитает метрики.

In [ ]:
subprocess.run([sys.executable, "evaluate.py", "extract", "--source", str(VIDEO),
                "--n", "25", "--stride", str(STRIDE)], check=True)

In [ ]:
gt = pd.read_csv(OUT / "ground_truth.csv")
if gt["true_count"].isna().all():
    print("Сначала заполните true_count в", OUT / "ground_truth.csv")
else:
    subprocess.run([sys.executable, "evaluate.py", "score", "--source", str(VIDEO)], check=True)
    display(Image(str(OUT / "evaluation.png")))

## 6. Выводы

- Предобученная YOLOv8n распознаёт легковые машины без дообучения: класс `car` есть в COCO, а модель весит около 6 МБ и работает в реальном времени на GPU.
- Трекинг ByteTrack присваивает каждой машине ID, поэтому можно считать не только «сколько сейчас», но и «сколько разных машин проехало или встало за запись».
- Сглаживание медианой за 1 секунду убирает мерцание: на отдельных кадрах детектор может пропустить машину, частично закрытую другой.
- Типичные ошибки — сильно перекрытые и очень мелкие машины на дальнем плане. Улучшить результат помогают больший `--imgsz` (1280), более тяжёлая модель (`yolov8s.pt`) или ROI, отсекающий дальний план.
- Применение: подсказки о свободных местах, статистика загрузки по часам, планирование парковочного пространства.